In [6]:
import os
import requests
from pathlib import Path
from shapely.geometry import Point, box
import geopandas as gpd
from tqdm import tqdm
import shutil
import json
import pandas as pd
import subprocess
from urllib.parse import urlparse

In [7]:

WFS_URL = "https://data.geopf.fr/private/wfs/wfs"
WFS_TYPENAME = "IGNF_LIDAR-HD_TA:nuage-dalle"  # point clouds layer
WFS_APIKEY = "interface_catalogue"  # public catalogue key used by IGN

def generate_lidar_filenames(lat, lon, buffer_km=5):
    pt = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs(epsg=2154).geometry[0]
    x_center, y_center = pt.x, pt.y
    x_min = int((x_center - buffer_km * 1000) // 1000)
    x_max = int((x_center + buffer_km * 1000) // 1000)
    y_min = int((y_center - buffer_km * 1000) // 1000)
    y_max = int((y_center + buffer_km * 1000) // 1000)

    return [
        f"LHD_FXX_{x:04d}_{y:04d}_PTS_C_LAMB93_IGN69.copc.laz"
        for x in range(x_min, x_max + 1)
        for y in range(y_min, y_max + 1)
    ]

def query_wfs_tiles(lat, lon, buffer_km=3):
    """Return a GeoDataFrame of tiles intersecting a square buffer around lat lon.
    The result has at least two useful fields: name and url."""
    pt = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs(2154).iloc[0]
    bbox = box(pt.x - buffer_km*1000, pt.y - buffer_km*1000, pt.x + buffer_km*1000, pt.y + buffer_km*1000)
    params = {
        "apikey": WFS_APIKEY,
        "SERVICE": "WFS",
        "REQUEST": "GetFeature",
        "VERSION": "2.0.0",
        "TYPENAMES": WFS_TYPENAME,
        "SRSNAME": "EPSG:2154",
        "BBOX": f"{bbox.bounds[0]},{bbox.bounds[1]},{bbox.bounds[2]},{bbox.bounds[3]},EPSG:2154",
        "OUTPUTFORMAT": "application/json",
        "COUNT": 5000,
    }
    r = requests.get(WFS_URL, params=params, timeout=60)
    r.raise_for_status()
    gdf = gpd.read_file(r.text)
    if "url" not in gdf.columns:
        raise RuntimeError("WFS response does not contain a url field")
    return gdf

def download_from_wfs_urls(gdf, dest_dir):
    """Download each tile from its exact WFS url into dest_dir.
    Returns a list of local filenames written."""
    dest = Path(dest_dir)
    dest.mkdir(parents=True, exist_ok=True)

    written = []
    with requests.Session() as s:
        for _, row in tqdm(gdf.iterrows(), total=len(gdf), desc="📥 Downloading tiles"):
            url = row["url"]
            # derive a safe filename from the URL
            fname = Path(urlparse(url).path).name
            if not fname:
                # fallback to name attribute if present
                fname = f"{row.get('name', 'tile')}.copc.laz"
            out = dest / fname
            if out.exists():
                written.append(fname)
                continue
            try:
                resp = s.get(url, stream=True, timeout=120)
                if resp.status_code == 404:
                    print(f"⛔ Tile not found at source: {fname}")
                    continue
                resp.raise_for_status()
                with open(out, "wb") as f_out:
                    for chunk in resp.iter_content(chunk_size=1024 * 256):
                        if chunk:
                            f_out.write(chunk)
                written.append(fname)
            except Exception as e:
                print(f"⚠️ Failed to download {fname}: {e}")
                continue
    return written

def download_if_missing(filenames, base_url, dest_dir):
    # kept for compatibility, not used once WFS is in place
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(exist_ok=True)

    for fname in tqdm(filenames, desc="📥 Downloading tiles"):
        url = base_url + fname
        dest_path = dest_dir / fname
        if dest_path.exists():
            continue
        try:
            response = requests.get(url, stream=True, timeout=30)
            if response.status_code == 404:
                print(f"⛔ Tile not found: {fname}")
                continue
            response.raise_for_status()
            with open(dest_path, "wb") as f_out:
                for chunk in response.iter_content(chunk_size=8192):
                    f_out.write(chunk)
        except Exception as e:
            print(f"⚠️ Failed to download {fname}: {e}")
            continue

def copy_selected_tiles(filenames, source_dir, target_dir):
    source_dir = Path(source_dir)
    target_dir = Path(target_dir)

    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    for fname in tqdm(filenames, desc="📤 Copying selected tiles"):
        src = source_dir / fname
        dst = target_dir / fname
        if src.exists():
            shutil.copy2(src, dst)
        else:
            print(f"⚠️ File not found in source: {fname}")

def process_lidar_site(lat, lon, name, buffer_km=3):
    # folders
    all_lidar_dir = "data/lidar_data/all_lidar"
    selected_lidar_dir = f"data/lidar_data/{name}/selected_lidar"

    print("🔍 Querying WFS for intersecting tiles...")
    gdf = query_wfs_tiles(lat, lon, buffer_km)

    if gdf.empty:
        print("⛔ No tiles returned by WFS for this area")
        return

    print("⬇️ Downloading exact tiles from their official URLs...")
    downloaded = download_from_wfs_urls(gdf, all_lidar_dir)

    # Build the list of filenames to copy from the URLs we just used
    filenames = [Path(urlparse(u).path).name or f"{n}.copc.laz"
                 for u, n in zip(gdf["url"], gdf.get("name", [None]*len(gdf)))]

    print("📁 Copying selected tiles to output folder...")
    copy_selected_tiles(filenames, all_lidar_dir, selected_lidar_dir)

    # PDAL pipeline
    pipeline = {
        "pipeline": [
            f"data/lidar_data/{name}/selected_lidar/*.laz",
            {
                "type": "writers.gdal",
                "filename": f"data/lidar_data/{name}/dsm_max.tif",
                "resolution": 1.0,
                "output_type": "max"
            },
            {
                "type": "writers.gdal",
                "filename": f"data/lidar_data/{name}/dsm_min.tif",
                "resolution": 1.0,
                "output_type": "min"
            }
        ]
    }

    Path(f"data/lidar_data/{name}").mkdir(parents=True, exist_ok=True)
    with open(f"data/lidar_data/{name}/pipeline_config.json", "w") as f:
        json.dump(pipeline, f, indent=2)

    print("📁 To create the tif files run: pdal pipeline", f"data/lidar_data/{name}/pipeline_config.json")
    print("✅ Done!")

In [8]:
cameras = pd.read_csv("site_data/cameras.csv")
cameras

,id,name,angle_of_view,elevation,lat,lon
0,70,haguenau-01,54.2,189.0,48.822194,7.779889
1,71,haguenau-02,54.2,189.0,48.822194,7.779889
2,72,bischwiller-01,54.2,162.0,48.772361,7.846500
3,73,bischwiller-02,54.2,167.0,48.772361,7.846500
4,74,betschdorf-01,54.2,167.0,48.897472,7.910694
5,76,germersheim-01,54.2,149.0,49.213108,8.366507
6,77,germersheim-02,54.2,149.0,49.213108,8.366507
7,40,croix-augas-01,54.2,150.0,48.426746,2.710876
8,41,croix-augas-02,54.2,150.0,48.426746,2.710876
9,63,nemours-01,54.2,106.0,48.260483,2.706383


In [9]:
id = 9
cam = cameras.iloc[id]
lat = cam["lat"]
lon = cam["lon"]
name = cam["name"][:-3]
name, lat, lon

('nemours', 48.2604833333, 2.7063833333)

In [10]:
process_lidar_site(lat,lon,name, buffer_km=3)

🔍 Querying WFS for intersecting tiles...
⬇️ Downloading exact tiles from their official URLs...


📥 Downloading tiles: 100%|██████████| 49/49 [05:21<00:00,  6.56s/it]


📁 Copying selected tiles to output folder...


📤 Copying selected tiles: 100%|██████████| 49/49 [00:03<00:00, 14.24it/s]

📁 To create the tif files run: pdal pipeline data/lidar_data/nemours/pipeline_config.json
✅ Done!
